# Task 134: pyxrf_fluor — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: X-ray fluorescence spectral fitting using pyXRF

**Paper**: PyXRF (Proc. SPIE publication)
**Repository**: https://github.com/NSLS-II/PyXRF

### Physical Background

Optical systems blur images via the Point Spread Function. Deconvolution inverts this to recover sharp detail.

### Forward Model

```
y = H * x + n  (PSF convolution)
```

### Inverse Problem

```
x_hat = argmin 1/2 ||H*x - y||^2 + lambda R(x)
```

### Performance Metrics
- **PSNR**: 40.00 dB (concentration), 55.25 dB (spectrum)
- **SSIM**: N/A
- **Evaluation**: XRF spectrum deconvolution — elemental concentration recovery (CC=0.9998, mean_RE=0.68%)

### Mathematical Formulation

A measured spectrum is modeled as a superposition of spectral profiles:

$$S(\nu) = \sum_{k=1}^{K} A_k \, P_k(\nu; \theta_k) + B(\nu) + \epsilon(\nu)$$

where $P_k$ is a peak profile (Gaussian/Lorentzian/Voigt), $B(\nu)$ is the baseline, and $\epsilon$ is noise.

**Voigt profile**: $V(\nu) = \int_{-\infty}^{\infty} G(\nu'; \sigma) \, L(\nu - \nu'; \gamma) \, d\nu'$

**Nonlinear least-squares fit**:
$$\hat{\theta} = \arg\min_\theta \sum_i \left(\frac{S_i^{\text{obs}} - S_i^{\text{model}}(\theta)}{\sigma_i}\right)^2$$


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
pyxrf_fluor - XRF Fluorescence Spectrum Deconvolution
=====================================================
Task: Deconvolve XRF fluorescence energy spectrum to quantify elemental composition
Repo: https://github.com/NSLS-II/PyXRF
Paper: Li et al., Proc. SPIE 2017, doi:10.1117/12.2272585

Inverse Problem:
    Given a measured XRF spectrum S(E), decompose it into individual elemental
    fluorescence line contributions to quantify elemental concentrations:
    S(E) = Σ_k c_k · L_k(E; E_k, σ_k) + B(E) + noise

Usage:
    /data/yjh/pyxrf_fluor_env/bin/python pyxrf_fluor_code.py
"""

import numpy as np
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`gaussian_peak`, `fwhm_to_sigma`, `build_xrf_model`

In [ ]:
# ═══════════════════════════════════════════════════════════
# 2. Forward Model: XRF Spectrum Generation
# ═══════════════════════════════════════════════════════════
def gaussian_peak(energy, center, amplitude, sigma):
    """Gaussian peak profile for detector-broadened XRF line."""
    return amplitude * np.exp(-0.5 * ((energy - center) / sigma) ** 2)

def fwhm_to_sigma(fwhm):
    """Convert FWHM to Gaussian sigma."""
    return fwhm / (2.0 * np.sqrt(2.0 * np.log(2.0)))

# ═══════════════════════════════════════════════════════════
# 4. Inverse Solver: XRF Spectrum Deconvolution
# ═══════════════════════════════════════════════════════════
def build_xrf_model(energy, elements, det_fwhm):
    """
    Build the multi-element XRF fitting model using lmfit.
    Model: Σ_k c_k · element_spectrum_k(E) + background(E)
    """
    det_sigma = fwhm_to_sigma(det_fwhm)
    
    # Pre-compute element basis spectra (unit concentration)
    basis_spectra = {}
    for element in elements:
        basis_spectra[element] = generate_element_spectrum(energy, element, 1.0, det_sigma)
    
    return basis_spectra

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
y = H * x + n  (PSF convolution)
```

Functions: `generate_element_spectrum`, `generate_background`, `forward_operator`, `load_or_generate_data`

In [ ]:
def generate_element_spectrum(energy, element, concentration, det_sigma):
    """
    Generate XRF spectrum for a single element.
    Each characteristic line is a Gaussian broadened by detector resolution.
    """
    spectrum = np.zeros_like(energy)
    if element not in XRF_LINES:
        return spectrum
    
    for line_name, line_energy, rel_intensity in XRF_LINES[element]:
        amplitude = concentration * rel_intensity
        spectrum += gaussian_peak(energy, line_energy, amplitude, det_sigma)
    
    return spectrum

def generate_background(energy, a=500.0, b=2.5, c=0.1):
    """
    Generate Bremsstrahlung background: exponential + scatter.
    B(E) = a * exp(-b*E) + c
    """
    return a * np.exp(-b * energy) + c

def forward_operator(energy, concentrations, det_fwhm=DETECTOR_FWHM):
    """
    Forward model: concentrations → XRF spectrum
    S(E) = Σ_k c_k · Σ_lines G(E; E_line, σ_det) · I_rel + B(E)
    """
    det_sigma = fwhm_to_sigma(det_fwhm)
    spectrum = np.zeros_like(energy)
    
    for element, conc in concentrations.items():
        spectrum += generate_element_spectrum(energy, element, conc, det_sigma)
    
    return spectrum

# ═══════════════════════════════════════════════════════════
# 3. Data Generation
# ═══════════════════════════════════════════════════════════
def load_or_generate_data():
    """
    Generate synthetic XRF spectrum with known elemental concentrations.
    Returns: (noisy_spectrum, gt_concentrations_array, metadata)
    """
    energy = np.arange(E_MIN, E_MAX, E_STEP)
    
    # Generate clean element spectra
    clean_signal = forward_operator(energy, GT_CONCENTRATIONS)
    
    # Generate background
    background = generate_background(energy)
    
    # Total clean spectrum
    clean_spectrum = clean_signal + background
    
    # Add Poisson-like noise (photon counting statistics)
    # Scale to realistic count rates
    scale_factor = 10.0  # counts per unit
    expected_counts = clean_spectrum * scale_factor
    noisy_counts = np.random.poisson(np.maximum(expected_counts, 1).astype(int)).astype(float)
    noisy_spectrum = noisy_counts / scale_factor
    
    # GT as array
    gt_array = np.array([GT_CONCENTRATIONS[el] for el in SAMPLE_ELEMENTS])
    
    metadata = {
        'energy': energy,
        'clean_signal': clean_signal,
        'background': background,
        'clean_spectrum': clean_spectrum,
        'elements': SAMPLE_ELEMENTS,
        'gt_concentrations': GT_CONCENTRATIONS,
        'det_fwhm': DETECTOR_FWHM,
    }
    
    print(f"[DATA] Generated XRF spectrum with {len(SAMPLE_ELEMENTS)} elements")
    print(f"[DATA] Energy range: {energy[0]:.1f} - {energy[-1]:.2f} keV")
    print(f"[DATA] GT concentrations: {GT_CONCENTRATIONS}")
    
    return noisy_spectrum, gt_array, metadata

## 5. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
x_hat = argmin 1/2 ||H*x - y||^2 + lambda R(x)
```

Functions: `reconstruct`

In [ ]:
def reconstruct(noisy_spectrum, metadata):
    """
    XRF spectrum deconvolution: recover elemental concentrations.
    
    Algorithm:
    1. Estimate and subtract background
    2. Build element basis spectra (unit concentration)
    3. Non-negative least squares fitting: S = Σ c_k · B_k + background
    4. Refine with lmfit for uncertainties
    """
    energy = metadata['energy']
    elements = metadata['elements']
    det_fwhm = metadata['det_fwhm']
    det_sigma = fwhm_to_sigma(det_fwhm)
    
    # Step 1: Build element basis spectra
    basis_spectra = build_xrf_model(energy, elements, det_fwhm)
    
    # Step 2: Construct design matrix [B_1, B_2, ..., B_k, bg_exp, bg_const]
    n_elements = len(elements)
    n_energy = len(energy)
    
    # Design matrix: each column is an element's basis spectrum
    A = np.zeros((n_energy, n_elements + 2))
    for i, element in enumerate(elements):
        A[:, i] = basis_spectra[element]
    
    # Background basis functions
    A[:, n_elements] = np.exp(-2.5 * energy)  # exponential background
    A[:, n_elements + 1] = np.ones(n_energy)  # constant background
    
    # Step 3: Non-negative least squares
    from scipy.optimize import nnls
    coeffs, residual = nnls(A, noisy_spectrum)
    
    fitted_concentrations = {}
    for i, element in enumerate(elements):
        fitted_concentrations[element] = float(coeffs[i])
    
    bg_amp = coeffs[n_elements]
    bg_const = coeffs[n_elements + 1]
    
    print(f"[RECON] NNLS fitted concentrations: {fitted_concentrations}")
    
    # Step 4: Refine with lmfit for better fit and uncertainties
    params = Parameters()
    for i, element in enumerate(elements):
        params.add(f'c_{element}', value=coeffs[i], min=0)
    params.add('bg_amp', value=bg_amp * 500, min=0)
    params.add('bg_decay', value=2.5, min=0.5, max=5.0)
    params.add('bg_const', value=bg_const, min=0)
    
    def residual_func(params, energy, data, elements, basis_spectra):
        model = np.zeros_like(energy)
        for element in elements:
            model += params[f'c_{element}'].value * basis_spectra[element]
        model += params['bg_amp'].value * np.exp(-params['bg_decay'].value * energy)
        model += params['bg_const'].value
        return (data - model)
    
    from lmfit import minimize as lm_minimize
    result = lm_minimize(residual_func, params, args=(energy, noisy_spectrum, elements, basis_spectra))
    
    # Extract refined concentrations
    refined_concentrations = {}
    for element in elements:
        refined_concentrations[element] = float(result.params[f'c_{element}'].value)
    
    print(f"[RECON] Refined concentrations: {refined_concentrations}")
    
    # Build reconstructed spectrum
    recon_spectrum = np.zeros_like(energy)
    for element in elements:
        recon_spectrum += refined_concentrations[element] * basis_spectra[element]
    recon_spectrum += result.params['bg_amp'].value * np.exp(-result.params['bg_decay'].value * energy)
    recon_spectrum += result.params['bg_const'].value
    
    # Build fitted background
    fitted_bg = result.params['bg_amp'].value * np.exp(-result.params['bg_decay'].value * energy) + result.params['bg_const'].value
    
    return {
        'nnls_concentrations': fitted_concentrations,
        'refined_concentrations': refined_concentrations,
        'recon_spectrum': recon_spectrum,
        'fitted_background': fitted_bg,
        'basis_spectra': basis_spectra,
        'fit_result': result,
    }

## 6. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `compute_metrics`, `visualize_results`

In [ ]:
# ═══════════════════════════════════════════════════════════
# 5. Evaluation Metrics
# ═══════════════════════════════════════════════════════════
def compute_metrics(gt_concentrations, recon_result, clean_spectrum, recon_spectrum):
    """Compute evaluation metrics for XRF deconvolution."""
    refined = recon_result['refined_concentrations']
    elements = list(gt_concentrations.keys())
    
    metrics = {}
    
    # Per-element concentration errors
    gt_array = []
    fitted_array = []
    rel_errors = []
    
    for el in elements:
        gt_val = gt_concentrations[el]
        fit_val = refined.get(el, 0.0)
        gt_array.append(gt_val)
        fitted_array.append(fit_val)
        re = abs(fit_val - gt_val) / gt_val * 100.0
        rel_errors.append(re)
        metrics[f'{el}_gt'] = gt_val
        metrics[f'{el}_fitted'] = round(fit_val, 4)
        metrics[f'{el}_RE_pct'] = round(re, 4)
    
    gt_array = np.array(gt_array)
    fitted_array = np.array(fitted_array)
    
    # Overall concentration metrics
    metrics['mean_RE_pct'] = float(np.mean(rel_errors))
    metrics['max_RE_pct'] = float(np.max(rel_errors))
    
    # Concentration CC
    cc = float(np.corrcoef(gt_array, fitted_array)[0, 1])
    metrics['concentration_CC'] = cc
    
    # Concentration R²
    ss_res = np.sum((gt_array - fitted_array) ** 2)
    ss_tot = np.sum((gt_array - np.mean(gt_array)) ** 2)
    metrics['concentration_R2'] = float(1 - ss_res / ss_tot) if ss_tot > 0 else 1.0
    
    # Concentration PSNR
    data_range = gt_array.max() - gt_array.min()
    mse_conc = np.mean((gt_array - fitted_array) ** 2)
    if mse_conc > 0:
        metrics['concentration_PSNR'] = float(10 * np.log10(data_range ** 2 / mse_conc))
    else:
        metrics['concentration_PSNR'] = float('inf')
    
    # Spectrum-level metrics
    data_range_spec = clean_spectrum.max() - clean_spectrum.min()
    mse_spec = np.mean((clean_spectrum - recon_spectrum) ** 2)
    if mse_spec > 0:
        metrics['spectrum_PSNR'] = float(10 * np.log10(data_range_spec ** 2 / mse_spec))
    else:
        metrics['spectrum_PSNR'] = float('inf')
    
    cc_spec = float(np.corrcoef(clean_spectrum, recon_spectrum)[0, 1])
    metrics['spectrum_CC'] = cc_spec
    
    rmse_spec = float(np.sqrt(mse_spec))
    metrics['spectrum_RMSE'] = rmse_spec
    
    return metrics

# ═══════════════════════════════════════════════════════════
# 6. Visualization
# ═══════════════════════════════════════════════════════════
def visualize_results(energy, noisy_spectrum, clean_spectrum, recon_result, 
                      gt_concentrations, metrics, save_path):
    """Generate 4-panel visualization for XRF deconvolution."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    recon_spectrum = recon_result['recon_spectrum']
    refined = recon_result['refined_concentrations']
    basis_spectra = recon_result['basis_spectra']
    fitted_bg = recon_result['fitted_background']
    elements = list(gt_concentrations.keys())
    
    colors = plt.cm.Set1(np.linspace(0, 1, len(elements)))
    
    # Panel 1: Noisy spectrum with individual element contributions (GT)
    ax1 = axes[0, 0]
    ax1.plot(energy, noisy_spectrum, 'k-', alpha=0.3, linewidth=0.5, label='Noisy data')
    ax1.plot(energy, clean_spectrum, 'b-', alpha=0.8, linewidth=1.0, label='Clean total')
    for i, el in enumerate(elements):
        det_sigma = fwhm_to_sigma(DETECTOR_FWHM)
        el_spec = generate_element_spectrum(energy, el, gt_concentrations[el], det_sigma)
        ax1.fill_between(energy, 0, el_spec, alpha=0.3, color=colors[i], label=f'{el} (GT)')
    ax1.set_xlabel('Energy (keV)')
    ax1.set_ylabel('Intensity (counts)')
    ax1.set_title('(a) GT XRF Spectrum — Element Contributions')
    ax1.legend(fontsize=7, ncol=2)
    ax1.set_xlim(E_MIN, E_MAX)
    
    # Panel 2: Fitted spectrum decomposition
    ax2 = axes[0, 1]
    ax2.plot(energy, noisy_spectrum, 'k-', alpha=0.3, linewidth=0.5, label='Noisy data')
    ax2.plot(energy, recon_spectrum, 'r-', alpha=0.8, linewidth=1.0, label='Fitted total')
    ax2.plot(energy, fitted_bg, 'g--', alpha=0.6, linewidth=1.0, label='Background')
    for i, el in enumerate(elements):
        el_spec = refined[el] * basis_spectra[el]
        ax2.fill_between(energy, 0, el_spec, alpha=0.3, color=colors[i], label=f'{el} (fit)')
    ax2.set_xlabel('Energy (keV)')
    ax2.set_ylabel('Intensity (counts)')
    psnr = metrics.get('spectrum_PSNR', 0)
    cc = metrics.get('spectrum_CC', 0)
    ax2.set_title(f'(b) Fitted XRF Decomposition — PSNR={psnr:.2f} dB, CC={cc:.4f}')
    ax2.legend(fontsize=7, ncol=2)
    ax2.set_xlim(E_MIN, E_MAX)
    
    # Panel 3: Residual
    ax3 = axes[1, 0]
    residual = noisy_spectrum - recon_spectrum
    ax3.plot(energy, residual, 'k-', alpha=0.5, linewidth=0.5)
    ax3.axhline(0, color='r', linestyle='--', alpha=0.5)
    ax3.fill_between(energy, -2*np.std(residual), 2*np.std(residual), alpha=0.1, color='blue')
    ax3.set_xlabel('Energy (keV)')
    ax3.set_ylabel('Residual')
    ax3.set_title(f'(c) Fit Residual — RMSE={metrics.get("spectrum_RMSE", 0):.4f}')
    ax3.set_xlim(E_MIN, E_MAX)
    
    # Panel 4: Concentration comparison (bar chart)
    ax4 = axes[1, 1]
    x = np.arange(len(elements))
    width = 0.35
    gt_vals = [gt_concentrations[el] for el in elements]
    fit_vals = [refined.get(el, 0) for el in elements]
    
    bars1 = ax4.bar(x - width/2, gt_vals, width, label='Ground Truth', color='steelblue', alpha=0.8)
    bars2 = ax4.bar(x + width/2, fit_vals, width, label='Fitted', color='coral', alpha=0.8)
    
    # Add RE labels
    for i, el in enumerate(elements):
        re = metrics.get(f'{el}_RE_pct', 0)
        ax4.annotate(f'{re:.1f}%', (x[i] + width/2, fit_vals[i] + 2), ha='center', fontsize=7)
    
    ax4.set_xlabel('Element')
    ax4.set_ylabel('Concentration (a.u.)')
    ax4.set_xticks(x)
    ax4.set_xticklabels(elements)
    mean_re = metrics.get('mean_RE_pct', 0)
    conc_cc = metrics.get('concentration_CC', 0)
    ax4.set_title(f'(d) Concentration Recovery — CC={conc_cc:.4f}, Mean RE={mean_re:.2f}%')
    ax4.legend()
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"[VIS] Saved visualization → {save_path}")

## 7. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
print("=" * 60)
print("  pyxrf_fluor — XRF Fluorescence Spectrum Deconvolution")
print("=" * 60)

### Ground Truth Construction

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# (a) Generate data
noisy_spectrum, gt_array, metadata = load_or_generate_data()
energy = metadata['energy']
clean_spectrum = metadata['clean_spectrum']

print(f"[DATA] Spectrum shape: {noisy_spectrum.shape}")

# (b) Run deconvolution
recon_result = reconstruct(noisy_spectrum, metadata)

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# (c) Evaluate
metrics = compute_metrics(GT_CONCENTRATIONS, recon_result, clean_spectrum,
                          recon_result['recon_spectrum'])

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
print(f"\n[EVAL] === Metrics ===")
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.6f}")
    else:
        print(f"  {k}: {v}")

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# (d) Save metrics
metrics_path = os.path.join(RESULTS_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"\n[SAVE] Metrics → {metrics_path}")

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# (e) Visualize
vis_path = os.path.join(RESULTS_DIR, "reconstruction_result.png")
visualize_results(energy, noisy_spectrum, clean_spectrum, recon_result,
                  GT_CONCENTRATIONS, metrics, vis_path)

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# (f) Save arrays
gt_conc_array = np.array([GT_CONCENTRATIONS[el] for el in SAMPLE_ELEMENTS])
fit_conc_array = np.array([recon_result['refined_concentrations'][el] for el in SAMPLE_ELEMENTS])
np.save(os.path.join(RESULTS_DIR, "ground_truth.npy"), gt_conc_array)
np.save(os.path.join(RESULTS_DIR, "reconstruction.npy"), fit_conc_array)

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
print("=" * 60)
print("  DONE — pyxrf_fluor XRF deconvolution complete")
print("=" * 60)

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 8. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **pyxrf_fluor**:

1. **Problem Setup**: Optical systems blur images via the Point Spread Function. Deconvolution inverts this to recover sharp detail.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=40.00 dB (concentration), 55.25 dB (spectrum), SSIM=N/A)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: PyXRF (Proc. SPIE publication)
- Repository: https://github.com/NSLS-II/PyXRF
